# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..','..')))
from ai_tools.tools import LLMQuery

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:12<00:00,  6.48s/it]


In [4]:
len(deals)

40

In [5]:
deals[10].describe()

'Title: Direktflüge USA / New York - mit Lufthansa nonstop von Berlin, München und Frankfurt inkl. Rückflug \nDetails: 345€ - SkyExplorer Spitzen Preise für Direktflüge mit der Lufthansa in die USA: Schon ab 345€ könnt ihr nonstop inkl. Rückflug von Berlin, München und Frankfurt nach New York fliegen. Hier findet ihr übersichtlich alle möglichen günstigen Terminkombinationen: Berlin - New York München - New York Frankfurt - New York New York City ist eine Stadt mit 5 Boroughs, die an der Mündung des Hudson River in den Atlantik liegt. Ihr dicht bevölkertes Herzstück bildet Manhattan, eines der bedeutendsten Hand\nFeatures: \nURL: https://www.mydealz.de/deals/direktfluge-usa-new-york-mit-lufthansa-nonstop-von-berlin-munchen-und-frankfurt-inkl-ruckflug-feb-dez-ab-345eur-2729687'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [6]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself in **english**, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [9]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [10]:


client = LLMQuery(system_prompt=SYSTEM_PROMPT, response_format=DealSelection)
response = client.query(make_user_prompt(deals))
print(response)

{
  "deals": [
    {
      "product_description": "This is a robust GIGABYTE Custom-Design graphics card intended for high-resolution WQHD gaming performance. It is equipped with 16GB of GDDR6 memory utilizing a 128-bit interface. The card features a base clock of 2690MHz and can boost up to 3320 MHz. Its design integrates the PCIe 5.0 x16 interface, making it an excellent upgrade choice even for older motherboards using PCIe 3.0 or 4.0 standards.",
      "price": 439.00,
      "url": "https://www.mydealz.de/deals/16gb-gigabyte-radeon-rx-9060-xt-gaming-oc-16g-16gb-gddr6-hdmi-2x-dp-fsr-4-grafikkarte-2729693"
    },
    {
      "product_description": "The WORX WG761E Nitro is a powerful, self-propelled cordless lawnmower built around an 80V battery system, which is supplied by four included 20V 4.0 Ah batteries. It features a robust brushless motor and offers a generous 51 cm cutting width, making it suitable for maintaining lawn areas up to 1000 m². Furthermore, it provides a versatile 

In [13]:
a = DealSelection.model_validate_json(response)
a

DealSelection(deals=[Deal(product_description='This is a robust GIGABYTE Custom-Design graphics card intended for high-resolution WQHD gaming performance. It is equipped with 16GB of GDDR6 memory utilizing a 128-bit interface. The card features a base clock of 2690MHz and can boost up to 3320 MHz. Its design integrates the PCIe 5.0 x16 interface, making it an excellent upgrade choice even for older motherboards using PCIe 3.0 or 4.0 standards.', price=439.0, url='https://www.mydealz.de/deals/16gb-gigabyte-radeon-rx-9060-xt-gaming-oc-16g-16gb-gddr6-hdmi-2x-dp-fsr-4-grafikkarte-2729693'), Deal(product_description='The WORX WG761E Nitro is a powerful, self-propelled cordless lawnmower built around an 80V battery system, which is supplied by four included 20V 4.0 Ah batteries. It features a robust brushless motor and offers a generous 51 cm cutting width, making it suitable for maintaining lawn areas up to 1000 m². Furthermore, it provides a versatile 3-in-1 function, allowing users to cho

In [16]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself in **english**, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: OilFox Smarte Füllstandsmessung für Heizöltanks (99€)
Details: 99€ Der FoxRadar ermöglicht die Füllstandsmessung per Radartechnologie – ganz einfach Außen am Tank angebracht, ohne die Notwendigkeit einer freien Tanköffnung. „Der intelligente OilFox-Sensor an Ihrem Heizöltank misst einmal täglich automatisch den aktuellen Füllstand und übermittelt diese Daten per 

In [17]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Radar-based external sensor for heating oil tanks that mounts on the outside of the tank and measures the oil level automatically once per day. It transmits current fill level, monthly consumption and historical data via Wi‑Fi to a companion app so you can monitor one or multiple tanks from a smartphone or tablet. The device requires no open tank access and is designed for easy installation and remote monitoring of heating oil inventory.', price=99.0, url='https://www.mydealz.de/deals/oilfox-smarte-fullstandsmessung-fur-heizoltanks-2729697'), Deal(product_description='10‑inch XIAOMI Scooter 4 Ultra electric scooter in black, designed for urban mobility with a compact wheel size and foldable frame. The model includes drivetrain and braking suited for city commuting and offers battery range and performance typical for the Scooter 4 Ultra line. It’s sold as a complete consumer e‑scooter ready for road use.', price=335.3, url='https://www.myde

In [24]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Radar-based external sensor for heating oil tanks that mounts on the outside of the tank and measures the oil level automatically once per day. It transmits current fill level, monthly consumption and historical data via Wi‑Fi to a companion app so you can monitor one or multiple tanks from a smartphone or tablet. The device requires no open tank access and is designed for easy installation and remote monitoring of heating oil inventory.
99.0
https://www.mydealz.de/deals/oilfox-smarte-fullstandsmessung-fur-heizoltanks-2729697

10‑inch XIAOMI Scooter 4 Ultra electric scooter in black, designed for urban mobility with a compact wheel size and foldable frame. The model includes drivetrain and braking suited for city commuting and offers battery range and performance typical for the Scooter 4 Ultra line. It’s sold as a complete consumer e‑scooter ready for road use.
335.3
https://www.mydealz.de/deals/xiaomi-scooter-4-ultra-e-scooter-10-zoll-black-mediamarkt-mwst-geschenkt-2729696

GIGABYTE

In [20]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [21]:
from agents.scanner_agent import ScannerAgent

In [22]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 40 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [23]:
result

DealSelection(deals=[Deal(product_description='Hisense RB440N4ACA is a tall (201 cm) frost-free refrigerator-freezer combination with about 336 L total capacity: ~238 L for refrigeration and ~98 L for freezing. It features NoFrost technology to eliminate manual defrosting, internal LED lighting, a Multi Airflow system for even cooling, a reversible door hinge, and a quiet operation around 39 dB. The exterior is stainless-steel look/silver and it’s rated energy class A with an annual consumption of ~109 kWh.', price=638.65, url='https://www.mydealz.de/deals/prime-hisense-rb440n4aca-kuhl-gefrier-kombinationhohe-201-cmkuhlen-178-lgefrieren-98-l-energieklasse-a-109kwh-pro-jahr-2729702'), Deal(product_description='Suzuki GSX-S1000 (2025 model) is a new 999 cm³ streetfighter motorcycle delivering 152 PS from a tuned four-cylinder derived from the GSX-R. It’s designed for aggressive on-road performance with sharp handling, sporty ergonomics, and modern electronics; the bike includes a 5-inch 

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [25]:
load_dotenv(override=True)

True

In [26]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [27]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with g
Pushover token found and starts with a


In [28]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [29]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [38]:
#from agents.messaging_agent import MessagingAgent
import importlib
# Modify 'mymodule.py' externally...
import agents
importlib.reload(agents.messaging_agent)
from agents.messaging_agent import MessagingAgent


agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [39]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
21:16:26 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= anthropic/claude-sonnet-4.5; provider = openrouter
INFO:LiteLLM:
LiteLLM completion() model= anthropic/claude-sonnet-4.5; provider = openrouter
21:16:29 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
